In [ ]:
from snowflake.snowpark.context import get_active_session
from datetime import datetime, timezone
import uuid
import re
import tempfile
import sqlparse

session       = get_active_session()
DB            = "DCF_RAWDATA"
RAW_SCHEMA    = "PUBLIC"
STAGE         = "@DCF_RAWDATA.RAWDATA.POSTGRES_STAGE"
CONTROL_TABLE = f"{DB}.{RAW_SCHEMA}.INCREMENTAL_LOAD_WATERMARK"
TMP_DIR       = tempfile.gettempdir()

TABLE_CONFIG = {
    "T_PERSONS":                    "PERSON_ID",
    "T_CASES":                      "CAS_ID",
    "T_MEDICATIONS":                "MED_ID",
    "T_MEDICATION_HEALTH_BEHAVIOR": "MHP_ID",
    "T_CUSTODIES":                  "CUS_ID",
    "T_PERSON_ORG_INVOLVEMENT":     "POI_ID",
    "T_PERSON_ROLE_ASSIGNMENTS":    "PRA1_ID",
}


def get_columns(table_name):
    rows = session.sql(f"""
        SELECT COLUMN_NAME
        FROM {DB}.INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{RAW_SCHEMA}'
          AND TABLE_NAME   = '{table_name}'
        ORDER BY ORDINAL_POSITION
    """).collect()
    return [r[0] for r in rows]


def get_max_loaded_ts():
    max_ts = None
    for table_name in TABLE_CONFIG:
        row = session.sql(f"""
            SELECT MAX(GREATEST(ADD_TS, COALESCE(MOD_TS, ADD_TS))) AS MAX_TS
            FROM {DB}.{RAW_SCHEMA}.{table_name}
        """).collect()[0]["MAX_TS"]
        if row is not None and (max_ts is None or row > max_ts):
            max_ts = row
    return max_ts


def update_watermark(file_name, file_ts, loaded_ts, processed_files):
    processed_files_str = ",".join(processed_files)
    session.sql(f"""
        MERGE INTO {CONTROL_TABLE} t
        USING (SELECT
            'INCREMENTAL_LOAD_RAWDATA'              AS PIPELINE_NAME,
            '{file_name}'                           AS LAST_FILE_NAME,
            '{file_ts}'::TIMESTAMP_NTZ              AS LAST_FILE_TS,
            '{loaded_ts}'::TIMESTAMP_NTZ            AS LAST_LOADED_TIMESTAMP,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ      AS LAST_RUN_TS,
            '{processed_files_str}'                 AS PROCESSED_FILES
        ) s ON t.PIPELINE_NAME = s.PIPELINE_NAME
        WHEN MATCHED THEN UPDATE SET
            t.LAST_FILE_NAME        = s.LAST_FILE_NAME,
            t.LAST_FILE_TS          = s.LAST_FILE_TS,
            t.LAST_LOADED_TIMESTAMP = s.LAST_LOADED_TIMESTAMP,
            t.LAST_RUN_TS           = s.LAST_RUN_TS,
            t.PROCESSED_FILES       = s.PROCESSED_FILES
        WHEN NOT MATCHED THEN INSERT
            (PIPELINE_NAME, LAST_FILE_NAME, LAST_FILE_TS, LAST_LOADED_TIMESTAMP, LAST_RUN_TS, PROCESSED_FILES)
        VALUES
            (s.PIPELINE_NAME, s.LAST_FILE_NAME, s.LAST_FILE_TS, s.LAST_LOADED_TIMESTAMP, s.LAST_RUN_TS, s.PROCESSED_FILES)
    """).collect()


# read watermark
wm_rows = session.sql(f"""
    SELECT LAST_FILE_TS, LAST_LOADED_TIMESTAMP, PROCESSED_FILES FROM {CONTROL_TABLE}
    WHERE PIPELINE_NAME = 'INCREMENTAL_LOAD_RAWDATA'
""").collect()

last_file_ts          = wm_rows[0]["LAST_FILE_TS"]          if wm_rows else None
last_loaded_timestamp = wm_rows[0]["LAST_LOADED_TIMESTAMP"] if wm_rows else None
raw_pf                = wm_rows[0]["PROCESSED_FILES"]       if wm_rows else None
processed_files       = set(raw_pf.split(",")) if raw_pf else set()

print(f"Watermark - Last File TS         : {last_file_ts}")
print(f"Watermark - Last Loaded Timestamp: {last_loaded_timestamp}")
print(f"Already Processed Files          : {processed_files}")

# list files from stage and skip already processed
stage_files = session.sql(f"LIST {STAGE}").collect()

new_files = []
for f in stage_files:
    name          = f["name"]
    last_modified = f["last_modified"]
    if not name.lower().endswith(".sql"):
        continue
    try:
        file_ts = datetime.strptime(last_modified.strip(), "%a, %d %b %Y %H:%M:%S %Z")
    except Exception:
        file_ts = datetime.strptime(last_modified.strip()[:24], "%a, %d %b %Y %H:%M:%S")

    file_base_name = name.split("/")[-1]
    if file_base_name in processed_files:
        print(f"  Skipping already processed: {file_base_name}")
        continue

    new_files.append((name, file_ts))

new_files.sort(key=lambda x: x[1])
print(f"\nNew .sql files to process: {len(new_files)}")
for nf in new_files:
    print(f"  {nf[0]}  ({nf[1]})")

if not new_files:
    print("No new files found. Nothing to process.")
else:
    job_id             = str(uuid.uuid4())
    start_time         = datetime.now(timezone.utc).replace(tzinfo=None)
    total_inserted     = 0
    total_updated      = 0
    stmt_errors        = 0
    stmt_error_details = []
    table_stmt_errors  = {}
    error_msg          = None
    table_stats        = []
    max_loaded_ts      = None

    try:
        for file_path, file_ts in new_files:
            file_name  = file_path.split("/")[-1]
            stage_path = f"{STAGE}/{file_name}"
            print(f"\nProcessing: {file_name}")

            session.file.get(stage_path, TMP_DIR)

            with open(f"{TMP_DIR}/{file_name}", "r", encoding="utf-8") as fh:
                sql_content = fh.read()

            # redirect inserts to TMP and updates to STAGE
            insert_sql = sql_content
            update_sql = sql_content
            for table_name in TABLE_CONFIG:
                insert_sql = insert_sql.replace(
                    f'DCF_RAWDATA.PUBLIC.{table_name}',
                    f'DCF_RAWDATA.PUBLIC.TMP_{table_name}'
                )
                update_sql = update_sql.replace(
                    f'DCF_RAWDATA.PUBLIC.{table_name}',
                    f'DCF_RAWDATA.PUBLIC.STAGE_{table_name}'
                )

            # extract PKs from update statements per table for targeted STAGE_ load
            update_pks = {table_name: [] for table_name in TABLE_CONFIG}
            for stmt in sqlparse.split(update_sql):
                stmt_clean = re.sub(r'--.*?(\n|$)', '', stmt).strip()
                if not stmt_clean.upper().startswith("UPDATE"):
                    continue
                for table_name, pk_col in TABLE_CONFIG.items():
                    if f"STAGE_{table_name}" in stmt_clean.upper():
                        match = re.search(
                            rf'WHERE\s+{pk_col}\s*=\s*(\d+)',
                            stmt_clean,
                            re.IGNORECASE
                        )
                        if match:
                            update_pks[table_name].append(match.group(1))

            # create empty TMP tables and targeted STAGE tables
            for table_name, pk_col in TABLE_CONFIG.items():
                session.sql(f"""
                    CREATE OR REPLACE TEMPORARY TABLE {DB}.{RAW_SCHEMA}.TMP_{table_name}
                    AS SELECT * FROM {DB}.{RAW_SCHEMA}.{table_name} WHERE 1=0
                """).collect()

                pks = update_pks.get(table_name, [])
                if pks:
                    pk_list = ", ".join(pks)
                    session.sql(f"""
                        CREATE OR REPLACE TEMPORARY TABLE {DB}.{RAW_SCHEMA}.STAGE_{table_name}
                        AS SELECT * FROM {DB}.{RAW_SCHEMA}.{table_name}
                        WHERE {pk_col} IN ({pk_list})
                    """).collect()
                else:
                    session.sql(f"""
                        CREATE OR REPLACE TEMPORARY TABLE {DB}.{RAW_SCHEMA}.STAGE_{table_name}
                        AS SELECT * FROM {DB}.{RAW_SCHEMA}.{table_name} WHERE 1=0
                    """).collect()

            # collect insert value blocks per table
            insert_batches = {table_name: [] for table_name in TABLE_CONFIG}
            for stmt in sqlparse.split(insert_sql):
                stmt_clean = re.sub(r'--.*?(\n|$)', '', stmt).strip()
                if not stmt_clean.upper().startswith("INSERT INTO"):
                    continue
                matched_table = next(
                    (t for t in TABLE_CONFIG if f"TMP_{t}" in stmt_clean.upper()),
                    None
                )
                if matched_table:
                    match = re.search(r'VALUES\s*(\(.*)', stmt_clean, re.IGNORECASE | re.DOTALL)
                    if match:
                        insert_batches[matched_table].append(match.group(1).strip())

            # insert rows into TMP per table then delete rows outside timestamp filter
            for table_name in TABLE_CONFIG:
                values = insert_batches.get(table_name, [])
                if not values:
                    continue
                tmp_table = f"{DB}.{RAW_SCHEMA}.TMP_{table_name}"
                columns   = get_columns(table_name)
                col_list  = ", ".join(columns)

                for value_block in values:
                    try:
                        session.sql(f"""
                            INSERT INTO {tmp_table} ({col_list})
                            VALUES {value_block}
                        """).collect()
                    except Exception as e:
                        print(f"Insert error on {table_name}: {e}")
                        stmt_errors += 1
                        err = f"INSERT error: {str(e)[:200]}"
                        stmt_error_details.append(f"[{file_name}] {table_name}: {err}")
                        table_stmt_errors.setdefault(table_name, []).append(err)

                # remove rows that don't meet timestamp filter
                if last_loaded_timestamp:
                    session.sql(f"""
                        DELETE FROM {tmp_table}
                        WHERE GREATEST(ADD_TS, COALESCE(MOD_TS, ADD_TS)) <= '{last_loaded_timestamp}'
                    """).collect()

            # run updates into STAGE tables
            for stmt in sqlparse.split(update_sql):
                stmt_clean = re.sub(r'--.*?(\n|$)', '', stmt).strip()
                if not stmt_clean.upper().startswith("UPDATE"):
                    continue
                matched_table = next(
                    (t for t in TABLE_CONFIG if f"STAGE_{t}" in stmt_clean.upper()),
                    "UNKNOWN"
                )
                try:
                    session.sql(stmt_clean).collect()
                except Exception as e:
                    print(f"Update error on {matched_table}: {e}")
                    stmt_errors += 1
                    err = f"UPDATE error: {str(e)[:200]}"
                    stmt_error_details.append(f"[{file_name}] {matched_table}: {err}")
                    table_stmt_errors.setdefault(matched_table, []).append(err)

            # merge TMP and STAGE into RAW, log audit and send notification per table
            for table_name, pk_col in TABLE_CONFIG.items():
                table_start_time = datetime.now(timezone.utc).replace(tzinfo=None)
                raw_table        = f"{DB}.{RAW_SCHEMA}.{table_name}"
                tmp_table        = f"{DB}.{RAW_SCHEMA}.TMP_{table_name}"
                stage_table      = f"{DB}.{RAW_SCHEMA}.STAGE_{table_name}"

                columns    = get_columns(table_name)
                col_list   = ", ".join(columns)
                set_clause = ", ".join(f"t.{c} = s.{c}" for c in columns if c != pk_col)
                val_clause = ", ".join(f"s.{c}" for c in columns)

                new_inserted = 0
                updated      = 0

                try:
                    result = session.sql(f"""
                        MERGE INTO {raw_table} t
                        USING (
                            SELECT * FROM {tmp_table}

                            UNION ALL

                            SELECT s.*
                            FROM {stage_table} s
                            WHERE NOT EXISTS (
                                SELECT 1 FROM {tmp_table} t2
                                WHERE t2.{pk_col} = s.{pk_col}
                            )
                        ) s
                        ON t.{pk_col} = s.{pk_col}
                        WHEN MATCHED
                            AND GREATEST(t.ADD_TS, COALESCE(t.MOD_TS, t.ADD_TS))
                             != GREATEST(s.ADD_TS, COALESCE(s.MOD_TS, s.ADD_TS))
                            THEN UPDATE SET {set_clause}
                        WHEN NOT MATCHED
                            THEN INSERT ({col_list}) VALUES ({val_clause})
                    """).collect()[0]

                    new_inserted = result["number of rows inserted"]
                    updated      = result["number of rows updated"]

                except Exception as e:
                    merge_error = f"MERGE error: {str(e)[:200]}"
                    stmt_errors += 1
                    stmt_error_details.append(f"[{file_name}] {table_name}: {merge_error}")
                    table_stmt_errors.setdefault(table_name, []).append(merge_error)
                    print(f"Merge error on {table_name}: {e}")

                table_end_time = datetime.now(timezone.utc).replace(tzinfo=None)

                total_inserted += new_inserted
                total_updated  += updated
                table_stats.append(
                    f"  [{file_name}] {table_name}: New={new_inserted}, Updated={updated}"
                )
                print(f"{table_name} -> New: {new_inserted} | Updated: {updated}")

                all_table_errors    = table_stmt_errors.get(table_name, [])
                table_error_count   = len(all_table_errors)
                table_error_message = "\n".join(all_table_errors)[:500] if all_table_errors else None
                table_status        = "FAILED" if all_table_errors else "SUCCESS"

                session.call(
                    "RETAIL_ETL_DB_NEW.AUDIT.SP_LOG_AUDIT",
                    job_id,
                    table_name,
                    "INCREMENTAL_LOAD",
                    table_start_time,
                    table_end_time,
                    new_inserted + updated,
                    table_error_count,
                    table_status,
                    table_error_message
                )

                table_message = (
                    f"File          : {file_name}\n"
                    f"Table         : {table_name}\n"
                    f"New Records   : {new_inserted}\n"
                    f"Updated       : {updated}\n"
                    f"Status        : {table_status}\n"
                    f"Processed At  : {table_end_time}"
                    + (f"\nErrors        :\n{table_error_message}" if table_error_message else "")
                )
                session.call(
                    "RETAIL_ETL_DB_NEW.UTIL.SP_SEND_PIPELINE_NOTIFICATION",
                    table_status, f"INCREMENTAL_LOAD - {table_name}", "RAWDATA", table_message
                )

            # update watermark after each file completes
            max_loaded_ts = get_max_loaded_ts()
            processed_files.add(file_name)
            update_watermark(file_name, file_ts, max_loaded_ts, processed_files)
            print(f"Watermark updated -> File: {file_name} | Last Loaded TS: {max_loaded_ts}")
            print(f"Total processed files so far: {len(processed_files)}")

        end_time      = datetime.now(timezone.utc).replace(tzinfo=None)
        audit_status  = "SUCCESS" if stmt_errors == 0 else "PARTIAL"
        audit_message = "\n".join(stmt_error_details) if stmt_error_details else None

        message = (
            f"Job ID              : {job_id}\n"
            f"Start Time          : {start_time}\n"
            f"End Time            : {end_time}\n"
            f"Files Processed     : {len(new_files)}\n"
            f"Last Loaded TS      : {max_loaded_ts}\n"
            f"Stmt Errors         : {stmt_errors}\n"
            f"\nTable Summary:\n"
            + "\n".join(table_stats) +
            f"\n\nTotal New     : {total_inserted}"
            f"\nTotal Updated : {total_updated}"
            + (f"\n\nStatement Errors:\n{audit_message}" if audit_message else "")
        )

        session.call(
            "RETAIL_ETL_DB_NEW.AUDIT.SP_LOG_AUDIT",
            job_id,
            "INCREMENTAL_LOAD_RAWDATA",
            "INCREMENTAL_LOAD",
            start_time,
            end_time,
            total_inserted + total_updated,
            stmt_errors,
            audit_status,
            audit_message
        )

        session.call(
            "RETAIL_ETL_DB_NEW.UTIL.SP_SEND_PIPELINE_NOTIFICATION",
            audit_status, "INCREMENTAL_LOAD", "RAWDATA", message
        )

    except Exception as e:
        error_msg  = str(e)
        end_time   = datetime.now(timezone.utc).replace(tzinfo=None)
        full_error = "\n".join(stmt_error_details) + f"\n\nFatal Error: {error_msg[:300]}" if stmt_error_details else error_msg[:300]

        message = (
            f"Job ID              : {job_id}\n"
            f"Start Time          : {start_time}\n"
            f"End Time            : {end_time}\n"
            f"Files               : {len(new_files)} found, partial processing\n"
            f"Stmt Errors         : {stmt_errors}\n"
            f"\nTable Summary:\n"
            + "\n".join(table_stats) +
            f"\n\nTotal New     : {total_inserted}"
            f"\nTotal Updated : {total_updated}"
            f"\n\nError Details:\n{full_error}"
        )

        session.call(
            "RETAIL_ETL_DB_NEW.AUDIT.SP_LOG_AUDIT",
            job_id,
            "INCREMENTAL_LOAD_RAWDATA",
            "INCREMENTAL_LOAD",
            start_time,
            end_time,
            total_inserted + total_updated,
            stmt_errors,
            "FAILED",
            full_error[:500]
        )

        session.call(
            "RETAIL_ETL_DB_NEW.UTIL.SP_SEND_PIPELINE_NOTIFICATION",
            "FAILED", "INCREMENTAL_LOAD", "RAWDATA", message
        )

        raise
